# Assignment 2 — Métricas y Validación de Modelos
## Unidad 2

**versión: 2025-1  |  modificado: 2026-04-06**

> 📝 **Modalidad:** Trabajo autónomo — practica a tu ritmo.

---

<div style="background-color:#F8F9FA; border:2px solid #AEB6BF;
     padding:12px 18px; border-radius:8px; margin:10px 0;">
<strong>🎓 Modo de uso:</strong> Este notebook es compartido por dos cursos.<br><br>
<span style="color:#2E86C1; font-weight:bold;">🔵 Pregrado</span>
— Trabaja el contenido general y los bloques azules. Los bloques amarillos
son opcionales y te darán contexto adicional.<br><br>
<span style="color:#B7950B; font-weight:bold;">🟡 Doctorado</span>
— Trabaja el contenido general y <em>ambos</em> bloques. Los bloques azules
te ofrecen la intuición; los amarillos, la formalización.
</div>

---

## Rúbrica Orientativa

> ⚠️ **Nota:** Esta rúbrica es orientativa. La nota final se asigna en la prueba escrita, no en este notebook.

<div style="background-color:#EBF5FB; border-left:5px solid #2E86C1;
     padding:14px 18px; border-radius:6px; margin:12px 0;">
<span style="color:#2E86C1; font-weight:bold; font-size:1em;">🔵 Pregrado — Competencias a demostrar</span><br><br>

- Calcular e interpretar accuracy, precision, recall, F1 y AUC-ROC en un problema real.
- Elegir la métrica adecuada según el contexto del problema (costos de error).
- Implementar validación cruzada estratificada con Pipeline (sin data leakage).
- Graficar e interpretar curvas ROC y de aprendizaje.
- Comparar dos modelos y justificar la elección.

<br>
<span style="font-size:0.85em; color:#5D6D7E;">→ ¿Quieres ver las competencias adicionales de doctorado? Continúa en el bloque 🟡.</span>
</div>

<div style="background-color:#FEFDE7; border-left:5px solid #D4AC0D;
     padding:14px 18px; border-radius:6px; margin:12px 0;">
<span style="color:#B7950B; font-weight:bold; font-size:1em;">🟡 Doctorado — Competencias adicionales</span><br><br>

- Todo lo anterior más:
- Derivar analíticamente el umbral óptimo bajo una función de costo asimétrica.
- Implementar y comparar CV simple vs Nested CV, cuantificando el sesgo de selección.
- Analizar la descomposición bias-varianza a través de curvas de aprendizaje.
- Conectar los resultados empíricos con la teoría de estimación de riesgo.
- Diseñar un experimento de ablación y formular una pregunta de investigación.

<br>
<span style="font-size:0.85em; color:#7D6608;">→ La intuición detrás de esta derivación está en el bloque 🔵 Pregrado.</span>
</div>

---
## Contexto del Problema

El **dataset Heart Disease** (Cleveland) contiene 303 pacientes con 13 características clínicas y una etiqueta binaria: `target=1` si el paciente tiene enfermedad cardíaca, `target=0` si no. El objetivo es construir y evaluar un clasificador que pueda asistir en el diagnóstico.

Este problema tiene **costos asimétricos**: no detectar una enfermedad cardíaca (FN) es mucho más grave que un diagnóstico erróneo hacia el lado positivo (FP, que lleva a más estudios).

**Restricciones:**
- No usar modelos de ensemble (RandomForest, XGBoost, etc.) — la clase de ensemble viene después.
- No mirar el conjunto de test hasta la sección de evaluación final.
- Cualquier transformación de features debe ir dentro de un Pipeline.

In [ ]:
# ── SETUP — NO MODIFICAR ──
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn import __version__ as sklearn_version
from sklearn.model_selection import (
    train_test_split, cross_val_score, StratifiedKFold,
    GridSearchCV, learning_curve, cross_validate
)
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report,
    roc_curve, roc_auc_score
)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
sns.set_theme(style='whitegrid', palette='colorblind')
plt.rcParams['figure.dpi'] = 100

# dataset: heart_disease_cleveland.csv  |  md5: d8c0a87bcb2a49be3c38c93d73f4c8b2
# Fuente: https://archive.ics.uci.edu/ml/datasets/Heart+Disease  (procesado)
# Carga directa desde URL pública del UCI ML Repository (versión preprocesada)
URL = 'https://raw.githubusercontent.com/dsrscientist/dataset1/master/heartdisease.csv'
try:
    df = pd.read_csv(URL)
    # Normalizar nombre de columna target (varía según la fuente)
    if 'target' not in df.columns and 'condition' in df.columns:
        df = df.rename(columns={'condition': 'target'})
    # Binarizar (algunas versiones tienen 0-4, queremos 0 vs 1+)
    if df['target'].max() > 1:
        df['target'] = (df['target'] > 0).astype(int)
    print(f'✅ Dataset cargado desde URL — shape: {df.shape}')
except Exception:
    # Fallback: generar dataset sintético con mismas propiedades
    from sklearn.datasets import make_classification
    X_syn, y_syn = make_classification(
        n_samples=303, n_features=13, n_informative=8,
        n_redundant=2, random_state=RANDOM_STATE, class_sep=0.8
    )
    cols = ['age','sex','cp','trestbps','chol','fbs','restecg',
            'thalach','exang','oldpeak','slope','ca','thal']
    df = pd.DataFrame(X_syn, columns=cols)
    df['target'] = y_syn
    print('⚠️  URL no disponible — usando dataset sintético con mismas propiedades')

# Separar features y target
FEATURE_COLS = [c for c in df.columns if c != 'target']
X = df[FEATURE_COLS]
y = df['target']

# Split estratificado (ANTES de cualquier transformación)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print(f'✅ Setup completo')
print(f'   numpy {np.__version__} · pandas {pd.__version__} · scikit-learn {sklearn_version}')
print(f'   Train: {X_train.shape} · Test: {X_test.shape}')
print(f'   Distribución target — train: {y_train.value_counts().to_dict()}')

---
## Sección 1 — EDA y Preprocesamiento

In [ ]:
# ━━━ SECCIÓN 1: EDA ━━━

# Explorar el dataset
print('=== Información general ===')
print(df.info())
print('\n=== Estadísticas descriptivas ===')
display(df.describe())
print('\n=== Valores nulos ===')
print(df.isnull().sum())
print('\n=== Distribución del target ===')
print(y.value_counts(normalize=True).round(3))

In [ ]:
# Visualización de distribuciones y correlaciones
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Distribución del target
y.value_counts().plot(kind='bar', ax=axes[0], color=['steelblue', 'coral'])
axes[0].set_title('Distribución del target', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Clase (0=sin enfermedad, 1=con enfermedad)', fontsize=11)
axes[0].set_ylabel('Frecuencia', fontsize=11)
axes[0].tick_params(rotation=0)

# Correlaciones con el target
corr_with_target = X_train.corrwith(y_train).sort_values(key=abs, ascending=False)
corr_with_target.plot(kind='bar', ax=axes[1], color='steelblue')
axes[1].set_title('Correlación features vs target (train set)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Feature', fontsize=11)
axes[1].set_ylabel('Correlación de Pearson', fontsize=11)
axes[1].axhline(0, color='black', linewidth=0.8)

plt.tight_layout()
plt.show()

---
## Sección 2 — Modelo Base: Evaluación con Múltiples Métricas

In [ ]:
# ━━━ SECCIÓN 2: MODELO BASE ━━━

# Pipeline base (Logistic Regression)
pipeline_base = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))
])
pipeline_base.fit(X_train, y_train)
y_pred_base = pipeline_base.predict(X_test)
y_prob_base = pipeline_base.predict_proba(X_test)[:, 1]

# Métricas
print('=== Métricas del Modelo Base (Regresión Logística) — Test Set ===')
print(classification_report(y_test, y_pred_base,
                             target_names=['Sin enfermedad', 'Con enfermedad']))

auc_base = roc_auc_score(y_test, y_prob_base)
print(f'AUC-ROC: {auc_base:.3f}')

# Visualizar matriz de confusión y curva ROC
cm_base = confusion_matrix(y_test, y_pred_base)
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

cm_norm = cm_base.astype(float) / cm_base.sum(axis=1, keepdims=True)
sns.heatmap(cm_norm, annot=cm_base, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Sin enfermedad', 'Con enfermedad'],
            yticklabels=['Sin enfermedad', 'Con enfermedad'],
            linewidths=0.5, cbar=False)
axes[0].set_xlabel('Predicción', fontsize=12)
axes[0].set_ylabel('Real', fontsize=12)
axes[0].set_title('Matriz de Confusión — Regresión Logística', fontsize=13, fontweight='bold')

fpr, tpr, _ = roc_curve(y_test, y_prob_base)
axes[1].plot(fpr, tpr, linewidth=2, label=f'Reg. Logística (AUC = {auc_base:.3f})')
axes[1].plot([0, 1], [0, 1], 'k--', linewidth=1, label='Azar')
axes[1].set_xlabel('FPR', fontsize=12)
axes[1].set_ylabel('TPR (Recall)', fontsize=12)
axes[1].set_title('Curva ROC', fontsize=13, fontweight='bold')
axes[1].legend(fontsize=10)

plt.tight_layout()
plt.show()

In [ ]:
# ─── TESTS DE SANIDAD — SECCIÓN 2 — NO MODIFICAR ───
print('=== Tests de Sanidad — Sección 2 ===')
try:
    acc_s2 = accuracy_score(y_test, y_pred_base)
    assert 0.0 <= acc_s2 <= 1.0, 'Accuracy fuera de rango [0,1]'
    print(f'✅ Accuracy en rango: {acc_s2:.3f}')
except Exception as e:
    print(f'❌ {e}')

try:
    assert 0.0 <= auc_base <= 1.0, 'AUC fuera de rango [0,1]'
    assert auc_base >= 0.5, f'AUC ({auc_base:.3f}) < 0.5 — ¿el modelo está peor que el azar?'
    print(f'✅ AUC válido y por encima del azar: {auc_base:.3f}')
except Exception as e:
    print(f'❌ {e}')

try:
    assert len(y_pred_base) == len(y_test), 'Predicciones y test no tienen el mismo tamaño'
    print(f'✅ Tamaño de predicciones correcto: {len(y_pred_base)}')
except Exception as e:
    print(f'❌ {e}')

try:
    assert pipeline_base.named_steps['scaler'].mean_ is not None, 'Scaler no fiteado'
    # Verificar que el scaler no fue fiteado en test
    scaler_mean = pipeline_base.named_steps['scaler'].mean_
    assert len(scaler_mean) == X_train.shape[1], 'El scaler fue fiteado con el número equivocado de features'
    print('✅ Scaler fiteado correctamente en train (no data leakage)')
except Exception as e:
    print(f'❌ {e}')

---
## Sección 3 — Comparación de Modelos con Validación Cruzada

In [ ]:
# ━━━ SECCIÓN 3: COMPARACIÓN CON CV ━━━

# Dos modelos candidatos
modelos = {
    'Reg. Logística': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))
    ]),
    'Árbol (depth=5)': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', DecisionTreeClassifier(max_depth=5, random_state=RANDOM_STATE))
    ])
}

CV_STRATEGY = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
METRICAS    = ['accuracy', 'f1', 'recall', 'precision', 'roc_auc']

resultados = {}
for nombre, modelo in modelos.items():
    scores = cross_validate(modelo, X_train, y_train,
                            cv=CV_STRATEGY, scoring=METRICAS)
    resultados[nombre] = {
        m: f"{scores[f'test_{m}'].mean():.3f} ± {scores[f'test_{m}'].std():.3f}"
        for m in METRICAS
    }

df_resultados = pd.DataFrame(resultados).T
print('=== Comparación de Modelos — Stratified 5-Fold CV (train set) ===')
display(df_resultados)

In [ ]:
# Visualizar comparación de F1 y AUC por modelo
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for i, (nombre, modelo) in enumerate(modelos.items()):
    for ax, metrica in zip(axes, ['f1', 'roc_auc']):
        scores = cross_val_score(modelo, X_train, y_train,
                                 cv=CV_STRATEGY, scoring=metrica)
        ax.boxplot(scores, positions=[i], widths=0.5,
                   patch_artist=True,
                   boxprops=dict(facecolor=f'C{i}', alpha=0.6))

for ax, metrica in zip(axes, ['F1-Score', 'AUC-ROC']):
    ax.set_xticks([0, 1])
    ax.set_xticklabels(list(modelos.keys()), fontsize=11)
    ax.set_ylabel(metrica, fontsize=12)
    ax.set_title(f'{metrica} por modelo (5-Fold CV)', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# ─── TESTS DE SANIDAD — SECCIÓN 3 — NO MODIFICAR ───
print('=== Tests de Sanidad — Sección 3 ===')
try:
    assert len(resultados) == 2, 'Deben compararse exactamente 2 modelos'
    print(f'✅ Dos modelos comparados: {list(resultados.keys())}')
except Exception as e:
    print(f'❌ {e}')

try:
    for nombre, modelo in modelos.items():
        assert isinstance(modelo, Pipeline), f'{nombre} debe ser un Pipeline'
    print('✅ Todos los modelos usan Pipeline (no data leakage en CV)')
except Exception as e:
    print(f'❌ {e}')

print('✅ Sanity checks OK — tu implementación parece correcta.')
print('   Recuerda revisar también las preguntas escritas de reflexión.')

---
## Sección 4 — Curvas de Aprendizaje y Diagnóstico

In [ ]:
# ━━━ SECCIÓN 4: CURVAS DE APRENDIZAJE ━━━

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
nombres_lista = list(modelos.keys())

for ax, nombre in zip(axes, nombres_lista):
    modelo = modelos[nombre]
    train_sizes_abs, train_sc, val_sc = learning_curve(
        modelo, X_train, y_train,
        train_sizes=np.linspace(0.15, 1.0, 8),
        cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE),
        scoring='f1', n_jobs=-1
    )
    train_mean = train_sc.mean(axis=1)
    val_mean   = val_sc.mean(axis=1)
    val_std    = val_sc.std(axis=1)

    ax.plot(train_sizes_abs, train_mean, 'o-', label='Train F1', linewidth=2)
    ax.plot(train_sizes_abs, val_mean,  's--', label='Val F1 (CV)', linewidth=2)
    ax.fill_between(train_sizes_abs,
                    val_mean - val_std, val_mean + val_std, alpha=0.15)
    ax.set_xlabel('Tamaño conjunto de entrenamiento', fontsize=12)
    ax.set_ylabel('F1-Score', fontsize=12)
    ax.set_title(f'Curva de Aprendizaje — {nombre}', fontsize=13, fontweight='bold')
    ax.legend(fontsize=10)
    ax.set_ylim([0.4, 1.05])

plt.tight_layout()
plt.show()

---
## Sección 5 — Análisis Diferenciado

<div style="background-color:#EBF5FB; border-left:5px solid #2E86C1;
     padding:14px 18px; border-radius:6px; margin:12px 0;">
<span style="color:#2E86C1; font-weight:bold; font-size:1em;">🔵 Pregrado</span><br><br>

**Análisis de métricas en contexto médico:**

Con base en los resultados de la Sección 3 y las curvas de aprendizaje:

1. ¿Qué modelo elegirías para un sistema de screening cardíaco? ¿Priorizas recall o precision? Justifica con los datos.

2. El médico jefe te dice: "prefiero revisar más pacientes sanos innecesariamente que perderme un enfermo". ¿Cómo ajustarías el umbral del modelo elegido? Experimenta en código.

3. Mirando las curvas de aprendizaje, ¿cree que conseguir más datos de pacientes mejoraría significativamente el modelo? ¿Cuántos necesitarías aproximadamente?

<br>
<span style="font-size:0.85em; color:#5D6D7E;">→ ¿Quieres ver la versión formal? Continúa en el bloque 🟡 Doctorado.</span>
</div>

<!-- RESPUESTA MODELO:
P1: En screening se prefiere Recall alto para no perder enfermos. / Reg. Logística generalmente da mejor calibración y recall en problemas de diagnóstico. / El Árbol puede ser más interpretable pero menos estable.
P2: Bajar el umbral (ej. de 0.5 a 0.3) aumenta el recall a costa de más FP. / Código: (y_prob >= 0.3).astype(int) y recalcular métricas. / La curva ROC muestra todos estos puntos de operación.
P3: Si train y val se acercan con más datos → más datos ayudan (alta varianza). / Si convergen en un valor alto → el modelo ya está bien. / Si ambas curvas son bajas → alto bias, más datos no ayudan.
-->

In [ ]:
# Espacio para exploración de umbral (Pregrado Pregunta 2)

# Ejemplo de cómo probar distintos umbrales:
umbrales = [0.3, 0.4, 0.5, 0.6, 0.7]
y_prob_train_val = pipeline_base.predict_proba(X_test)[:, 1]

print('=== Efecto del umbral en Recall y Precision ===')
print(f'{"Umbral":>8} {"Precision":>10} {"Recall":>8} {"F1":>8}')
print('-' * 40)
for theta in umbrales:
    y_pred_theta = (y_prob_train_val >= theta).astype(int)
    p = precision_score(y_test, y_pred_theta, zero_division=0)
    r = recall_score(y_test, y_pred_theta, zero_division=0)
    f = f1_score(y_test, y_pred_theta, zero_division=0)
    print(f'{theta:>8.2f} {p:>10.3f} {r:>8.3f} {f:>8.3f}')

**Respuesta 1 (Pregrado):**

*(escribe aquí)*

**Respuesta 2 (Pregrado):**

*(escribe aquí)*

**Respuesta 3 (Pregrado):**

*(escribe aquí)*

<div style="background-color:#FEFDE7; border-left:5px solid #D4AC0D;
     padding:14px 18px; border-radius:6px; margin:12px 0;">
<span style="color:#B7950B; font-weight:bold; font-size:1em;">🟡 Doctorado</span><br><br>

**Análisis formal y extensión:**

4. Dado que $C_{FN} = 5000$ USD (tratamiento no administrado) y $C_{FP} = 200$ USD (prueba innecesaria), deriva analíticamente el umbral óptimo $\theta^* = C_{FP} / (C_{FP} + C_{FN})$ y verifícalo empíricamente en los datos de test.

5. Implementa **Nested CV** (loop externo: StratifiedKFold(5), loop interno: GridSearchCV sobre `C` en logspace(-3, 3, 10)) y compara el F1 reportado con el F1 de la CV simple. Cuantifica el sesgo de selección.

6. A partir de las curvas de aprendizaje, ¿qué componente de la descomposición bias-varianza domina en cada modelo? Apoya tu respuesta con el gap train-val observado.

<br>
<span style="font-size:0.85em; color:#7D6608;">→ La intuición detrás de esta derivación está en el bloque 🔵 Pregrado.</span>
</div>

<!-- RESPUESTA MODELO:
P4: theta* = 200/(200+5000) ≈ 0.038. / Con ese umbral, casi todos los positivos son detectados (Recall muy alto) a costa de muchos FP. / Verificar: (y_prob >= theta_opt).astype(int) y calcular costo total = C_FN*FN + C_FP*FP.
P5: Nested CV genera F1 ligeramente menor que CV simple. / La diferencia (sesgo) suele ser 0.01-0.05 dependiendo del tamaño del grid. / Con grid de 10 valores, el sesgo es measurable pero no catastrófico en este dataset.
P6: RL → bajo bias (train y val altos), varianza moderada (gap pequeño). / Árbol → alta varianza (gap grande), bajo bias en train (train F1 ≈ 1.0). / Regularizar el árbol o usar más datos reduciría la varianza del árbol.
-->

In [ ]:
# [PhD] Espacio para los análisis de doctorado

# Pregunta 4: umbral óptimo bajo función de costo
C_FN = 5000
C_FP = 200
theta_opt = C_FP / (C_FP + C_FN)
print(f'Umbral óptimo teórico: θ* = {C_FP}/({C_FP}+{C_FN}) = {theta_opt:.4f}')

# TODO [PhD]: Verifica empíricamente el umbral óptimo calculando el costo total
# para cada umbral en np.linspace(0.01, 0.99, 50) y graficando costo vs umbral.
# Tu código aquí:


In [ ]:
# [PhD] Pregunta 5: Nested CV
# TODO [PhD]: Implementa Nested CV y compara con CV simple
# Tu código aquí:


**Respuesta 4 [PhD]:**

*(escribe aquí)*

**Respuesta 5 [PhD]:**

*(escribe aquí)*

**Respuesta 6 [PhD]:**

*(escribe aquí)*

---
## ✅ Sección 6 — Autoevaluación

### Pregrado
- [ ] Calculé e interpreté correctamente accuracy, precision, recall, F1 y AUC
- [ ] Usé un Pipeline con StandardScaler en todas las evaluaciones
- [ ] Comparé dos modelos con Stratified 5-Fold CV
- [ ] Grafiqué e interpreté la curva de aprendizaje
- [ ] Respondí las 3 preguntas de reflexión
- [ ] El notebook ejecuta de arriba a abajo sin errores

### Doctorado (adicional)
- [ ] Derivé y verifiqué el umbral óptimo bajo función de costo
- [ ] Implementé Nested CV y cuantifiqué el sesgo de selección
- [ ] Analicé la descomposición bias-varianza desde las curvas de aprendizaje
- [ ] Respondí las 3 preguntas de análisis crítico

---

**¿Qué fue lo más difícil de este assignment?**

*(escribe aquí)*

**¿Qué queda sin entender o quisieras explorar más?**

*(escribe aquí)*